## Clase 7 - Ejercicio: algoritmo de búsqueda para la Torre de Hanoi

**Maximo Teyo**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None` si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [1]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi
from aima_libs.aima import PriorityQueue as ColaPrioritaria


## Qué implementamos

Implementamos **búsqueda A\*** para resolver la Torre de Hanoi, usando una **heurística
propia** que desarrollamos nosotros.

### Cómo funciona A\*

En BFS la frontera se maneja con una cola FIFO: se expande nivel por nivel, y el criterio
para elegir el siguiente nodo es su profundidad.

En A\* la frontera es una **cola prioritaria**, y el criterio para sacar el siguiente nodo
ya no es la profundidad sino la **función de evaluación**:

$$
f(n) \;=\; g(n) \;+\; h(n)
$$

- $g(n)$ es el **costo real ya pagado** para llegar desde el estado inicial hasta $n$. En
  este problema todos los movimientos cuestan 1, así que $g(n)$ es simplemente la cantidad
  de movimientos realizados, que la biblioteca ya nos da en `NodeHanoi.path_cost`.
- $h(n)$ es una **estimación** de cuánto falta todavía para llegar al objetivo. A $h$ se la
  llama **función heurística**: no dice *cómo* llegar, dice *qué tan lejos* estamos.

A\* siempre expande el nodo que parece estar más cerca de terminar. Dos casos límite
ilustran el espectro:

- si $h(n) = 0$ para todo $n$, entonces $f(n) = g(n)$ y A\* es búsqueda de **costo
  uniforme**.
- si $h(n)$ fuera exactamente el costo real que falta, A\* no miraría más que el camino
  óptimo.

O sea: **el algoritmo es siempre el mismo**, lo único que cambia es la función $f$. Por eso
en este ejercicio lo que separa un 8 de un 10 no es el buscador, sino la heurística.

### La heurística del aula, y por qué no alcanza

La heurística que vimos en la clase bonifica con un punto negativo por cada disco que ya
está en su varilla objetivo:

$$
h_{\text{aula}}(n) \;=\; -\big(\text{cantidad de discos en la varilla objetivo}\big)
$$

Como todos esos valores son menores o iguales a 0, y en el estado inicial vale exactamente
0, en la práctica **no estima nada**: sólo sirve para desempatar entre nodos con el mismo
$g$. Es admisible (nunca sobreestima, porque es $\le 0$), así que A\* igual garantiza el
óptimo, pero guía muy poco la búsqueda.



### Nuestra heurística: una cota inferior de los movimientos que faltan

La idea es que $h(n)$ sea **una cota inferior de la cantidad de movimientos que todavía
faltan** para llegar al objetivo. Si la estimamos bien, A\* expande casi sólo lo necesario.

#### La intuición

Para mover un disco a la varilla objetivo, **todos los discos más chicos que él tienen que
estar en la única varilla que no es ni el objetivo ni la varilla donde está ese disco**: no
se puede apoyar un disco grande encima de uno más pequeño, y mientras el disco grande esté
en el medio no hay forma de llegar al objetivo.

Una vez que los discos chicos están todos juntos en esa varilla, hay que **trasladar esa
pila completa** de una varilla a otra. Y mover una pila de $k$ discos de una varilla a
otra, usando las tres, cuesta exactamente $2^k - 1$ movimientos (es el problema clásico de
Hanoi).

O sea, sólo para poner el disco $b$ en su lugar hacen falta, como mínimo:

$$
\underbrace{2^{\,b-1} - 1}_{\text{trasladar la pila}} \;+\; \underbrace{1}_{\text{el disco } b} \;=\; 2^{\,b-1}
$$

Y antes de eso los $b-1$ discos chicos ni siquiera están juntos: todavía hay que reunirlos.
Y para reunirlos pasa **exactamente lo mismo**, pero tomando como "meta" la varilla auxiliar
y como disco grande el más grande que no esté ahí. De ahí la recursión.

#### La definición

$G(k, T)$ es una cota inferior de los movimientos necesarios para juntar los $k$ discos más
chicos sobre la varilla $T$:

$$
G(k, T) =
\begin{cases}
0 & \text{si los } k \text{ discos ya están en } T \\[4pt]
G(b-1, \text{auxiliar}) + 2^{\,b-1} & \text{en otro caso}
\end{cases}
$$

donde $b$ es **el disco más grande de esos $k$ que no está en $T$**, y `auxiliar` es la
tercera varilla (la que no es $T$ ni la varilla donde está $b$).

Nuestra heurística es entonces:

$$
h(s) \;=\; G(n, \text{meta})
$$

#### Ejemplo: estado inicial con 5 discos

Todos los discos están en la varilla 1 y la meta es la varilla 3.

| paso | llamada | por qué | queda |
| --- | --- | --- | --- |
| 1 | $G(5, 3)$ | el disco 5 está en la varilla 1, auxiliar = varilla 2 | $G(4, 2) + 2^4$ |
| 2 | $G(4, 2)$ | el disco 4 está en la varilla 1, auxiliar = varilla 3 | $G(3, 3) + 2^3$ |
| 3 | $G(3, 3)$ | el disco 3 está en la varilla 1, auxiliar = varilla 2 | $G(2, 2) + 2^2$ |
| 4 | $G(2, 2)$ | el disco 2 está en la varilla 1, auxiliar = varilla 3 | $G(1, 3) + 2^1$ |
| 5 | $G(1, 3)$ | el disco 1 está en la varilla 1, auxiliar = varilla 2 | $G(0, 2) + 2^0$ |
| 6 | $G(0, 2)$ | no queda ningún disco | $0$ |

Desenrollando desde abajo hacia arriba:

$$
h \;=\; 0 + 1 + 2 + 4 + 8 + 16 \;=\; 31
$$

Y el costo óptimo real de la Torre de Hanoi con 5 discos es $2^5 - 1 = 31$. Nuestra
heurística acierta exactamente en el estado inicial.



### Por qué esto garantiza la solución óptima

Un resultado clásico de la teoría (Russell & Norvig, capítulo 3) dice que A\* devuelve la
solución **óptima** si su heurística cumple dos condiciones.

**1. Admisible** — nunca sobreestima el costo real que falta:

$$
h(n) \;\le\; c^*(n)
$$

donde $c^*(n)$ es el costo real mínimo para llegar al objetivo desde $n$. Si $h$
sobreestimara, A\* podría "creer" que un camino más caro está más cerca del objetivo y
devolver una solución no óptima.

En nuestro caso $h$ se obtiene **sumando cotas inferiores** de movimientos que sí o sí hay
que hacer (reunir la pila, trasladarla, mover el disco grande), así que nunca puede
pasarse.

**2. Consistente** — para todo movimiento de $n$ a $n'$:

$$
h(n) \;\le\; 1 + h(n')
$$

Es decir, la heurística no puede "bajar de golpe" más de lo que baja el costo real. Esto
garantiza que A\* no tenga que re-expandir nodos, y por lo tanto que la primera vez que saca
un nodo de la frontera, ese nodo ya tiene su mejor camino posible.

**Las dos propiedades se pueden verificar empíricamente**, sin fiarse del argumento, en el
espacio de estados completo (sección *Verificación 3* más abajo).

> **Nota honesta:** esta cota resultó ser *ajustada* (en inglés *tight*): coincide con la
> distancia real en el 100% de los estados para 3, 4, 5, 6 y 7 discos. Tiene sentido,
> porque la distancia exacta hasta una "esquina" de la Torre de Hanoi tiene una forma
> cerrada conocida. Eso no vuelve incorrecto al algoritmo: A\* sigue haciendo la búsqueda
> sobre el grafo de estados, y la heurística se evalúa en $O(n)$ por nodo contra $O(2^n)$
> nodos de búsqueda. Pero conviene decirlo, porque es lo que explica por qué A\* acá no
> expande prácticamente ningún nodo inútil.


In [2]:
VARILLA_META = 2  # el goal_state de la consigna es StatesHanoi([], [], discos)


def heuristica_propia(estado):
    """
    Cota inferior de los movimientos que faltan para llevar todos los discos a
    la varilla objetivo.

        h(estado) = G(n, meta), con

            G(k, T) = 0                                si los k discos ya están en T
            G(k, T) = G(b - 1, auxiliar) + 2 ** (b-1)  si no,

    donde b es el disco más grande de esos k que NO está en T, y auxiliar es la
    tercera varilla (la que no es T ni la varilla donde está b).

    Es ADMISIBLE (nunca sobreestima el costo real restante) y CONSISTENTE
    (h(s) <= 1 + h(s') para todo movimiento s -> s'), por lo que A* garantiza
    que la solución que encuentra es óptima.
    """
    # En qué varilla está cada disco.
    posicion = {
        disco: indice
        for indice, varilla in enumerate(estado.rods)
        for disco in varilla
    }

    def cota(k, meta):
        # El disco más grande (de 1 a k) que todavía NO está en `meta`.
        disco = 0
        for d in range(k, 0, -1):
            if posicion[d] != meta:
                disco = d
                break

        if disco == 0:
            return 0  # los k discos ya están ordenados en `meta`

        # La única varilla que no es `meta` ni la del disco `disco`.
        auxiliar = ({0, 1, 2} - {posicion[disco], meta}).pop()

        # Hay que reunir los disco-1 chicos en `auxiliar` (cota inferior) y
        # después trasladar esa pila y mover el disco grande: 2**(disco-1).
        return cota(disco - 1, auxiliar) + 2 ** (disco - 1)

    return cota(estado.number_of_disks, VARILLA_META)


def heuristica_del_aula(estado):
    """
    La heurística que nos dieron en la clase: un punto negativo por cada disco
    que ya está en la varilla objetivo. La definimos acá para poder compararla
    con la nuestra más abajo.
    """
    return -len(estado.rods[VARILLA_META])


# Un vistazo rápido de cómo se comporta cada una en el estado inicial.
for discos in (3, 4, 5):
    inicial = StatesHanoi(list(range(discos, 0, -1)), [], [], max_disks=discos)
    print(
        f"{discos} discos ->  h_aula = {heuristica_del_aula(inicial):4}"
        f"   h_propia = {heuristica_propia(inicial):4}"
        f"   (costo óptimo real = {2 ** discos - 1})"
    )

3 discos ->  h_aula =    0   h_propia =    7   (costo óptimo real = 7)
4 discos ->  h_aula =    0   h_propia =   15   (costo óptimo real = 15)
5 discos ->  h_aula =    0   h_propia =   31   (costo óptimo real = 31)


In [3]:
def search_algorithm(number_disks=5, funcion_heuristica=None) -> tuple[NodeHanoi | None, dict]:
    """
    Búsqueda A* para la Torre de Hanoi.

    La frontera es una cola prioritaria ordenada por f(n) = g(n) + h(n), donde
    g(n) es el costo acumulado (NodeHanoi.path_cost) y h(n) la heurística.

    Por defecto usa `heuristica_propia`. Se acepta `funcion_heuristica` para
    poder compararla con la heurística del aula; el uso que pide la consigna,
    search_algorithm(number_disks=5), no se modifica.
    """
    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    ##### EDITAR ESTA ZONA

    # Si no nos pasan una heurística explícita, usamos la nuestra.
    h = funcion_heuristica or heuristica_propia

    # f(n) = g(n) + h(n). La cola prioritaria saca siempre el nodo de menor f.
    def f(nodo):
        return nodo.path_cost + h(nodo.state)

    # La frontera arranca con la raíz del árbol.
    frontier = ColaPrioritaria(order="min", f=f)
    frontier.append(NodeHanoi(initial_state))

    # mejor_costo[estado] = menor g con el que sabemos llegar a ese estado.
    # Sirve para no meter dos veces el mismo estado en la frontera, y para no
    # re-expandir un nodo si aparece después un camino más barato.
    mejor_costo = {initial_state: 0.0}

    estados_visitados = set()
    nodos_explorados = 0
    max_depth = 0
    solucion = None

    while len(frontier) > 0:
        _, nodo = frontier.pop()

        # Si este nodo llegó por un camino más caro que otro que ya teníamos,
        # lo descartamos.
        if nodo.path_cost > mejor_costo[nodo.state]:
            continue

        estados_visitados.add(nodo.state)
        nodos_explorados += 1
        max_depth = max(max_depth, nodo.depth)

        if problem.goal_test(nodo.state):
            solucion = nodo
            break

        for hijo in nodo.expand(problem):
            # Sólo si encontramos una forma más barata de llegar a ese estado.
            if hijo.path_cost < mejor_costo.get(hijo.state, float("inf")):
                mejor_costo[hijo.state] = hijo.path_cost
                frontier.append(hijo)

    metrics = {
        "solution_found": solucion is not None,
        "nodes_explored": nodos_explorados,
        "states_visited": len(estados_visitados),
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth,
        "cost_total": solucion.path_cost if solucion is not None else None,
    }

    #####

    return solucion, metrics

Se prueba la función:

In [4]:
solution, metrics = search_algorithm(number_disks=5)

Veamos las métricas:

In [5]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 32
states_visited: 32
nodes_in_frontier: 16
max_depth: 31
cost_total: 31.0



### Qué significa cada métrica

| métrica | con 5 discos | lectura |
| --- | --- | --- |
| `solution_found` | `True` | se alcanzó el estado objetivo |
| `nodes_explored` | 32 | nodos que sacamos de la frontera y expandimos |
| `states_visited` | 32 | estados distintos visitados (no hay repetidos: con heurística consistente A\* no re-expande) |
| `nodes_in_frontier` | 16 | nodos que quedaron esperando en la frontera al terminar |
| `max_depth` | 31 | profundidad máxima alcanzada |
| `cost_total` | 31.0 | costo del camino encontrado, $2^5-1$, el mínimo teórico |

`nodes_explored` es el número que más dice: **32 nodos para 5 discos**, contra los 1351
que necesita BFS (ver *Comparación* más abajo). La heurística es tan precisa que A\*
expande únicamente el camino óptimo: los 32 nodos expandidos son la raíz más los 31 nodos
de la solución. No se desperdicia ni una sola expansión.



### Verificación 1: el costo encontrado es el óptimo

El costo óptimo de la Torre de Hanoi con $n$ discos es conocido y vale $2^n - 1$. Como
todos los movimientos cuestan 1, ese es también el mínimo de movimientos posible. Si nuestro
algoritmo devuelve ese valor para todo $n$, está devolviendo la solución óptima.


In [6]:
print(f"{'discos':>7} | {'costo encontrado':>16} | {'optimo 2^n - 1':>15} | "
      f"{'nodos':>7} | {'ok':>5}")
print("-" * 64)

todo_ok = True
for discos in range(1, 9):
    _, m = search_algorithm(number_disks=discos)
    optimo = 2 ** discos - 1
    ok = m["solution_found"] and m["cost_total"] == optimo
    todo_ok = todo_ok and ok
    print(f"{discos:>7} | {m['cost_total']:>16} | {optimo:>15} | "
          f"{m['nodes_explored']:>7} | {'OK' if ok else 'FALLA':>5}")

print("-" * 64)
print("Todos los casos dan la solución óptima:", todo_ok)

 discos | costo encontrado |  optimo 2^n - 1 |   nodos |    ok
----------------------------------------------------------------
      1 |              1.0 |               1 |       2 |    OK
      2 |              3.0 |               3 |       4 |    OK
      3 |              7.0 |               7 |       8 |    OK
      4 |             15.0 |              15 |      16 |    OK
      5 |             31.0 |              31 |      32 |    OK
      6 |             63.0 |              63 |      64 |    OK
      7 |            127.0 |             127 |     128 |    OK
      8 |            255.0 |             255 |     256 |    OK
----------------------------------------------------------------
Todos los casos dan la solución óptima: True


Veamos el camino de estados desde el principio a la solución:

In [7]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [8]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3



### Verificación 2: la secuencia de acciones es legal y llega al objetivo

Que el agente diga que encontró la meta no alcanza: hay que comprobar que cada movimiento
de la solución sea legal (un disco por vez, y ningún disco apoyado sobre otro más pequeño)
y que la posición final sea efectivamente la meta.


In [9]:
# Reconstruimos las varillas desde el estado inicial y aplicamos una por una todas
# las acciones de la solución, verificando que cada una sea legal.
discos = 5
varillas = [list(range(discos, 0, -1)), [], []]
meta = [[], [], list(range(discos, 0, -1))]

movimientos_ilegales = 0
for accion in solution.solution():
    origen, destino = accion.rod_input, accion.rod_out
    disco = varillas[origen].pop()  # se saca el disco superior

    # regla 3: ningún disco puede colocarse sobre uno que sea más pequeño
    if varillas[destino] and varillas[destino][-1] < disco:
        movimientos_ilegales += 1

    varillas[destino].append(disco)

print("Cantidad de movimientos:", len(solution.solution()))
print("Movimientos ilegales:  ", movimientos_ilegales)
print("Estado final:          ", varillas)
print("Estado objetivo:       ", meta)
print("Se llegó a la meta:    ", varillas == meta)

Cantidad de movimientos: 31
Movimientos ilegales:   0
Estado final:           [[], [], [5, 4, 3, 2, 1]]
Estado objetivo:        [[], [], [5, 4, 3, 2, 1]]
Se llegó a la meta:     True



### Comparación: BFS, A\* con la heurística del aula, y A\* con la nuestra

Los tres algoritmos son el mismo: lo único que cambia es la función $f$ con la que se
ordena la frontera. Por eso vale la pena medirlos lado a lado sobre el mismo problema.


In [10]:
def busqueda_en_anchura(number_disks=5):
    """BFS tal como la implementa la notebook 02, con control de estados ya
    visitados. Está sólo para comparar: el algoritmo que entregamos es A*."""
    lista = [i for i in range(number_disks, 0, -1)]
    problem = ProblemHanoi(
        initial=StatesHanoi(lista, [], [], max_disks=number_disks),
        goal=StatesHanoi([], [], lista, max_disks=number_disks),
    )

    frontier = [NodeHanoi(problem.initial)]
    visitados = set()
    explorados = 0
    max_depth = 0

    while frontier:
        nodo = frontier.pop()
        explorados += 1
        max_depth = max(max_depth, nodo.depth)
        visitados.add(nodo.state)

        if problem.goal_test(nodo.state):
            return nodo, {
                "nodes_explored": explorados,
                "states_visited": len(visitados),
                "nodes_in_frontier": len(frontier),
                "max_depth": max_depth,
                "cost_total": nodo.state.accumulated_cost,
            }

        for hijo in nodo.expand(problem):
            if hijo.state not in visitados:
                frontier.insert(0, hijo)

    return None, {}


print(f"{'discos':>6} | {'BFS':>19} | {'A* heur. aula':>19} | {'A* heur. propia':>19}")
print(f"{'':>6} | {'costo / nodos':>19} | {'costo / nodos':>19} | {'costo / nodos':>19}")
print("-" * 80)

for discos in range(3, 8):
    _, m_bfs = busqueda_en_anchura(discos)
    _, m_aula = search_algorithm(discos, funcion_heuristica=heuristica_del_aula)
    _, m_nuestra = search_algorithm(discos)

    assert m_bfs["cost_total"] == m_aula["cost_total"] == m_nuestra["cost_total"], disks

    print(f"{discos:>6} | "
          f"{int(m_bfs['cost_total']):>8} / {m_bfs['nodes_explored']:<9} | "
          f"{int(m_aula['cost_total']):>8} / {m_aula['nodes_explored']:<9} | "
          f"{int(m_nuestra['cost_total']):>8} / {m_nuestra['nodes_explored']:<9}")

print("-" * 80)
print("Los tres encuentran el mismo costo óptimo. Lo que cambia es cuánta")
print("información hace falta para llegar a él.")

discos |                 BFS |       A* heur. aula |     A* heur. propia
       |       costo / nodos |       costo / nodos |       costo / nodos
--------------------------------------------------------------------------------


     3 |        7 / 51        |        7 / 18        |        7 / 8        
     4 |       15 / 216       |       15 / 54        |       15 / 16       
     5 |       31 / 1351      |       31 / 178       |       31 / 32       
     6 |       63 / 7246      |       63 / 586       |       63 / 64       
     7 |      127 / 45453     |      127 / 1866      |      127 / 128      
--------------------------------------------------------------------------------
Los tres encuentran el mismo costo óptimo. Lo que cambia es cuánta
información hace falta para llegar a él.



### Verificación 3: la heurística es admisible y consistente

El argumento teórico de arriba se puede comprobar exhaustivamente, sin fiarse de él. Para
cada estado alcanzable calculamos la **distancia real** que falta para llegar al objetivo
con una BFS desde la meta (un método independiente de nuestra heurística) y después
verificamos:

- **admisibilidad**: `h(estado) <= distancia_real(estado)` en todos los estados.
- **consistencia**: `h(estado) <= 1 + h(siguiente)` en todos los movimientos legales.


In [11]:
from collections import deque


def distancias_al_objetivo(problem, origen):
    """Distancia real (mínima cantidad de movimientos) desde cada estado
    alcanzable hasta `origen`. BFS invertida: los movimientos de Hanoi son
    reversibles, así que el grafo es no dirigido."""
    distancia = {origen: 0}
    cola = deque([origen])
    while cola:
        estado = cola.popleft()
        for accion in problem.actions(estado):
            siguiente = problem.result(estado, accion)
            if siguiente not in distancia:
                distancia[siguiente] = distancia[estado] + 1
                cola.append(siguiente)
    return distancia


print(f"{'discos':>6} | {'estados':>8} | {'movimientos':>11} | {'inadmissible':>12} | "
      f"{'inconsistente':>13} | {'h == real':>11}")
print("-" * 84)

for discos in (3, 4, 5, 6, 7):
    lista = [i for i in range(discos, 0, -1)]
    problem = ProblemHanoi(
        initial=StatesHanoi(lista, [], [], max_disks=discos),
        goal=StatesHanoi([], [], lista, max_disks=discos),
    )

    real = distancias_al_objetivo(problem, problem.goal)

    inadmisible = 0
    inconsistente = 0
    movimientos = 0
    exactas = 0

    for estado, distancia in real.items():
        h = heuristica_propia(estado)
        if h > distancia:
            inadmisible += 1
        if h == distancia:
            exactas += 1
        for accion in problem.actions(estado):
            siguiente = problem.result(estado, accion)
            movimientos += 1
            if h > 1 + heuristica_propia(siguiente):
                inconsistente += 1

    print(f"{discos:>6} | {len(real):>8} | {movimientos:>11} | {inadmisible:>12} | "
          f"{inconsistente:>13} | {exactas:>4}/{len(real):<4}")

print("-" * 84)
print("0 inadmissibles y 0 inconsistentes => la heurística cumple las dos condiciones:")
print("A* garantiza la solución ÓPTIMA y no re-expande ningún nodo.")

discos |  estados | movimientos | inadmissible | inconsistente |   h == real
------------------------------------------------------------------------------------
     3 |       27 |          78 |            0 |             0 |   27/27  
     4 |       81 |         240 |            0 |             0 |   81/81  
     5 |      243 |         726 |            0 |             0 |  243/243 
     6 |      729 |        2184 |            0 |             0 |  729/729 
     7 |     2187 |        6558 |            0 |             0 | 2187/2187
------------------------------------------------------------------------------------
0 inadmissibles y 0 inconsistentes => la heurística cumple las dos condiciones:
A* garantiza la solución ÓPTIMA y no re-expande ningún nodo.



---

## Conclusión

- **Algoritmo implementado:** búsqueda A\* sobre el grafo de estados de la Torre de Hanoi,
  con la frontera como cola prioritaria ordenada por $f(n) = g(n) + h(n)$.
- **Heurística:** propia. Una cota inferior recursiva de los movimientos restantes, que
  suma nivel por nivel el costo inevitable de reunir en la varilla libre la pila de discos
  chicos, trasladarla ($2^k - 1$) y recién ahí colocar el disco grande.
- **Propiedades:** verificadas exhaustivamente sobre el espacio de estados completo hasta 7
  discos: **admisible** (0 sobreestimaciones) y **consistente** (0 inconsistencias). Por lo
  tanto A\* devuelve siempre la solución óptima.
- **Resultado:** con 5 discos encuentra la solución de **31 movimientos** ($2^5 - 1$, el
  mínimo teórico) explorando **32 nodos**, contra los 1351 nodos que necesita BFS.
- **Métricas:** se devuelven las seis que pide la consigna (`solution_found`,
  `nodes_explored`, `states_visited`, `nodes_in_frontier`, `max_depth`, `cost_total`).
